# GNN-Based Phishing Account Detection in Ethereum
## Model Architecture & Training

Implementation of:
1. **GAE_PDNA** - Graph AutoEncoder with Pathfinder Discovery Network layers
2. **MagNet Link Prediction** - Directed graph link prediction
3. **Embedding Baselines** - DeepWalk, Node2Vec
4. **Classifiers** - AdaBoost, Random Forest, SVM, Naive Bayes, Logistic Regression

Based on: *Ratra et al., "Graph neural network based phishing account detection in Ethereum", The Computer Journal, 2024*

In [ ]:
# ============================================================
# 1. INSTALLATION
# ============================================================
!pip install -q torch torch-geometric torch-sparse torch-scatter
!pip install -q imbalanced-learn node2vec gensim scikit-learn
!pip install -q matplotlib seaborn
print('All packages installed.')

In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from torch_geometric.data import Data
from torch_geometric.nn import PDNConv, GCNConv
from torch_geometric.nn.norm import GraphNorm
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import negative_sampling

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import OneClassSVM
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, precision_score,
    recall_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Load Dataset

Upload `node_features_labeled.csv` and `transactions_clean.csv` to Colab.

In [ ]:
# ============================================================
# 3. UPLOAD DATA FILES
# ============================================================
from google.colab import files

print('Upload node_features_labeled.csv and transactions_clean.csv')
uploaded = files.upload()

In [ ]:
# ============================================================
# 4. BUILD PyG GRAPH WITH NODE + EDGE ATTRIBUTES
# ============================================================
nodes_df = pd.read_csv('node_features_labeled.csv')
edges_df = pd.read_csv('transactions_clean.csv')

print(f'Nodes: {len(nodes_df)}, Edges: {len(edges_df)}')
print(f'Phishing: {nodes_df["label"].sum()}, '
      f'Non-phishing: {(nodes_df["label"]==0).sum()}')

node_index = {addr: i for i, addr in enumerate(nodes_df['address'])}

# Node features: in_degree, out_degree, total_in, total_out, balance
node_feat_cols = ['in_degree', 'out_degree', 'total_in', 'total_out', 'balance']
X_raw = nodes_df[node_feat_cols].values.astype(np.float32)

# Log transform balance (paper: reduce range of balance values)
X_raw[:, 4] = np.log1p(np.abs(X_raw[:, 4])) * np.sign(X_raw[:, 4])

# Normalize each feature independently
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)
x = torch.tensor(X_scaled, dtype=torch.float)

# Labels
y = torch.tensor(nodes_df['label'].values, dtype=torch.long)

# Edge index and edge attributes (timeStamp, value, blockNumber)
edge_src, edge_dst = [], []
edge_attrs = []

for _, row in edges_df.iterrows():
    if row['from'] in node_index and row['to'] in node_index:
        edge_src.append(node_index[row['from']])
        edge_dst.append(node_index[row['to']])
        edge_attrs.append([
            float(row['timeStamp']),
            float(row['value']),
            float(row['blockNumber'])
        ])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

# Normalize edge attributes
edge_attr_np = np.array(edge_attrs, dtype=np.float32)
edge_scaler = MinMaxScaler()
edge_attr_scaled = edge_scaler.fit_transform(edge_attr_np)
edge_attr = torch.tensor(edge_attr_scaled, dtype=torch.float)

# Build PyG Data object
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
data = data.to(device)

print(f'\nPyG Graph constructed:')
print(f'  Nodes: {data.num_nodes}')
print(f'  Edges: {data.num_edges}')
print(f'  Node features: {data.num_node_features}')
print(f'  Edge features: {data.edge_attr.shape[1]}')
print(f'  Labels distribution: {data.y.unique(return_counts=True)}')

## 5. Model 1: GAE_PDNA

Graph AutoEncoder using Pathfinder Discovery Network (PDN) layers.

**Architecture (from paper):**
- 4 blocks of: `PDNConv -> PReLU -> GraphNorm`
- Final `PDNConv` layer
- Output embedding dimension: 15
- Edge hidden channels: 6
- Decoder: Inner product decoder for link reconstruction

In [ ]:
# ============================================================
# 5. GAE_PDNA MODEL DEFINITION
# ============================================================
class PDNBlock(nn.Module):
    """Single block: PDNConv -> PReLU -> GraphNorm"""
    def __init__(self, in_channels, out_channels, edge_dim, edge_hidden):
        super().__init__()
        self.conv = PDNConv(
            in_channels=in_channels,
            out_channels=out_channels,
            edge_dim=edge_dim,
            hidden_channels=edge_hidden
        )
        self.act = nn.PReLU()
        self.norm = GraphNorm(out_channels)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv(x, edge_index, edge_attr)
        x = self.act(x)
        x = self.norm(x)
        return x


class GAE_PDNA_Encoder(nn.Module):
    """Encoder: 4 PDN blocks + final PDNConv (paper Section 5.1.4)"""
    def __init__(self, in_channels, hidden_channels, out_channels,
                 edge_dim, edge_hidden, num_blocks=4):
        super().__init__()
        self.blocks = nn.ModuleList()

        # First block: in_channels -> hidden_channels
        self.blocks.append(
            PDNBlock(in_channels, hidden_channels, edge_dim, edge_hidden)
        )
        # Remaining blocks: hidden_channels -> hidden_channels
        for _ in range(num_blocks - 1):
            self.blocks.append(
                PDNBlock(hidden_channels, hidden_channels, edge_dim, edge_hidden)
            )
        # Final PDNConv: hidden_channels -> out_channels
        self.final_conv = PDNConv(
            in_channels=hidden_channels,
            out_channels=out_channels,
            edge_dim=edge_dim,
            hidden_channels=edge_hidden
        )

    def forward(self, x, edge_index, edge_attr):
        for block in self.blocks:
            x = block(x, edge_index, edge_attr)
        x = self.final_conv(x, edge_index, edge_attr)
        return x


class InnerProductDecoder(nn.Module):
    """Decoder: reconstructs adjacency via inner product of embeddings."""
    def forward(self, z, edge_index):
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=1)


class GAE_PDNA(nn.Module):
    """Full Graph AutoEncoder with PDN encoder + inner product decoder."""
    def __init__(self, in_channels, hidden_channels=32, out_channels=15,
                 edge_dim=3, edge_hidden=6, num_blocks=4):
        super().__init__()
        self.encoder = GAE_PDNA_Encoder(
            in_channels, hidden_channels, out_channels,
            edge_dim, edge_hidden, num_blocks
        )
        self.decoder = InnerProductDecoder()

    def encode(self, x, edge_index, edge_attr):
        return self.encoder(x, edge_index, edge_attr)

    def decode(self, z, edge_index):
        return self.decoder(z, edge_index)

    def forward(self, x, edge_index, edge_attr):
        z = self.encode(x, edge_index, edge_attr)
        return z

print('GAE_PDNA model defined.')

In [ ]:
# ============================================================
# 6. GAE_PDNA TRAINING
# ============================================================
# Random link split: 70% train, 10% val, 20% test (paper Section 6.2.2)
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    is_undirected=False,
    add_negative_train_samples=True,
    split_labels=True
)

train_data, val_data, test_data = transform(data.cpu())
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

# Model parameters (paper: in=5 node feats, out=15, edge_hidden=6)
IN_CHANNELS = data.num_node_features  # 5
HIDDEN_CHANNELS = 32
OUT_CHANNELS = 15
EDGE_DIM = 3
EDGE_HIDDEN = 6
NUM_BLOCKS = 4
EPOCHS = 200
LR = 0.01

model_gae = GAE_PDNA(
    in_channels=IN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=OUT_CHANNELS,
    edge_dim=EDGE_DIM,
    edge_hidden=EDGE_HIDDEN,
    num_blocks=NUM_BLOCKS
).to(device)

optimizer_gae = Adam(model_gae.parameters(), lr=LR)


def gae_loss(z, pos_edge_index, neg_edge_index):
    """Binary cross-entropy reconstruction loss."""
    pos_scores = model_gae.decode(z, pos_edge_index)
    neg_scores = model_gae.decode(z, neg_edge_index)

    pos_loss = F.binary_cross_entropy_with_logits(
        pos_scores, torch.ones_like(pos_scores)
    )
    neg_loss = F.binary_cross_entropy_with_logits(
        neg_scores, torch.zeros_like(neg_scores)
    )
    return pos_loss + neg_loss


def train_gae_epoch():
    model_gae.train()
    optimizer_gae.zero_grad()

    z = model_gae.encode(
        train_data.x, train_data.edge_index, train_data.edge_attr
    )

    pos_edge = train_data.pos_edge_label_index
    neg_edge = train_data.neg_edge_label_index

    loss = gae_loss(z, pos_edge, neg_edge)
    loss.backward()
    optimizer_gae.step()
    return loss.item()


@torch.no_grad()
def eval_gae(split_data):
    model_gae.eval()
    z = model_gae.encode(
        split_data.x, split_data.edge_index, split_data.edge_attr
    )
    pos_edge = split_data.pos_edge_label_index
    neg_edge = split_data.neg_edge_label_index
    loss = gae_loss(z, pos_edge, neg_edge)
    return loss.item()


# Training loop (paper: 200 epochs)
print('Training GAE_PDNA...')
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    t_loss = train_gae_epoch()
    v_loss = eval_gae(val_data)
    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}')

test_loss = eval_gae(test_data)
print(f'\nFinal Test Loss: {test_loss:.4f}')
print(f'Final Train Loss: {train_losses[-1]:.4f}')
print(f'Final Val Loss: {val_losses[-1]:.4f}')

In [ ]:
# ============================================================
# 7. GAE_PDNA LOSS CURVES
# ============================================================
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('GAE_PDNA Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. EXTRACT NODE EMBEDDINGS FROM GAE_PDNA
# ============================================================
model_gae.eval()
with torch.no_grad():
    embeddings_gae = model_gae.encode(
        data.x, data.edge_index, data.edge_attr
    ).cpu().numpy()

labels_np = data.y.cpu().numpy()

print(f'GAE_PDNA embeddings shape: {embeddings_gae.shape}')
print(f'  -> {embeddings_gae.shape[0]} nodes x {embeddings_gae.shape[1]} features')

## 9. Model 2: MagNet Link Prediction

MagNet uses a complex Hermitian matrix (magnetic Laplacian) for directed graphs.

**Parameters (from paper):**
- Hidden channels: 8
- q parameter: trainable, initialized to 0.15
- Dropout: 0.8
- Training: 100 epochs

In [ ]:
# ============================================================
# 9. MAGNET LINK PREDICTION
# ============================================================
from torch_geometric.nn import MagNet

class MagNetLinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels=8, out_channels=2,
                 num_layers=2, q=0.15, dropout=0.8):
        super().__init__()
        self.magnet = MagNet(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_layers=num_layers,
            q=q,
            dropout=dropout,
            trainable_q=True
        )

    def forward(self, x, edge_index):
        # MagNet returns real and imaginary parts
        out_real, out_imag = self.magnet(x, edge_index)
        return out_real, out_imag

    def decode(self, z_real, z_imag, edge_index):
        src, dst = edge_index
        z = torch.cat([z_real, z_imag], dim=1)
        score = (z[src] * z[dst]).sum(dim=1)
        return score


model_magnet = MagNetLinkPredictor(
    in_channels=data.num_node_features,
    hidden_channels=8,
    out_channels=2,
    q=0.15,
    dropout=0.8
).to(device)

optimizer_magnet = Adam(model_magnet.parameters(), lr=0.01)
MAGNET_EPOCHS = 100

print('Training MagNet Link Prediction...')

for epoch in range(1, MAGNET_EPOCHS + 1):
    model_magnet.train()
    optimizer_magnet.zero_grad()

    z_real, z_imag = model_magnet(
        train_data.x, train_data.edge_index
    )

    pos_edge = train_data.pos_edge_label_index
    neg_edge = train_data.neg_edge_label_index

    pos_scores = model_magnet.decode(z_real, z_imag, pos_edge)
    neg_scores = model_magnet.decode(z_real, z_imag, neg_edge)

    scores = torch.cat([pos_scores, neg_scores])
    labels_link = torch.cat([
        torch.ones(pos_scores.size(0)),
        torch.zeros(neg_scores.size(0))
    ]).to(device)

    loss = F.binary_cross_entropy_with_logits(scores, labels_link)
    loss.backward()
    optimizer_magnet.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')

print('MagNet training complete.')

In [ ]:
# ============================================================
# 10. MAGNET EVALUATION (Link Prediction)
# ============================================================
model_magnet.eval()

with torch.no_grad():
    z_real, z_imag = model_magnet(test_data.x, test_data.edge_index)

    pos_edge = test_data.pos_edge_label_index
    neg_edge = test_data.neg_edge_label_index

    pos_scores = torch.sigmoid(model_magnet.decode(z_real, z_imag, pos_edge))
    neg_scores = torch.sigmoid(model_magnet.decode(z_real, z_imag, neg_edge))

    scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
    true_labels = np.concatenate([
        np.ones(pos_scores.size(0)),
        np.zeros(neg_scores.size(0))
    ])
    pred_labels = (scores > 0.5).astype(int)

print('MagNet Link Prediction Results (Test):')
print(classification_report(true_labels, pred_labels, digits=2))

magnet_auc = roc_auc_score(true_labels, scores)
print(f'AUC Score: {magnet_auc:.4f}')

## 11. Embedding Baselines: DeepWalk & Node2Vec

Random walk-based embedding methods for comparison.

In [ ]:
# ============================================================
# 11. DEEPWALK & NODE2VEC EMBEDDINGS
# ============================================================
import networkx as nx
from node2vec import Node2Vec

# Build NetworkX graph from edge index
edge_list = data.edge_index.cpu().numpy()
G_nx = nx.DiGraph()
G_nx.add_nodes_from(range(data.num_nodes))
for i in range(edge_list.shape[1]):
    G_nx.add_edge(edge_list[0, i], edge_list[1, i])

print(f'NetworkX graph: {G_nx.number_of_nodes()} nodes, '
      f'{G_nx.number_of_edges()} edges')

# --- DeepWalk (p=1, q=1, equivalent to unbiased random walk) ---
print('\nGenerating DeepWalk embeddings...')
dw_model = Node2Vec(
    G_nx, dimensions=15, walk_length=30,
    num_walks=200, workers=2, p=1, q=1
)
dw_fit = dw_model.fit(window=10, min_count=1, batch_words=4)

embeddings_deepwalk = np.zeros((data.num_nodes, 15))
for node_id in range(data.num_nodes):
    if str(node_id) in dw_fit.wv:
        embeddings_deepwalk[node_id] = dw_fit.wv[str(node_id)]

print(f'DeepWalk embeddings shape: {embeddings_deepwalk.shape}')

# --- Node2Vec (p=2, q=0.5, biased random walk) ---
print('\nGenerating Node2Vec embeddings...')
n2v_model = Node2Vec(
    G_nx, dimensions=15, walk_length=30,
    num_walks=200, workers=2, p=2, q=0.5
)
n2v_fit = n2v_model.fit(window=10, min_count=1, batch_words=4)

embeddings_node2vec = np.zeros((data.num_nodes, 15))
for node_id in range(data.num_nodes):
    if str(node_id) in n2v_fit.wv:
        embeddings_node2vec[node_id] = n2v_fit.wv[str(node_id)]

print(f'Node2Vec embeddings shape: {embeddings_node2vec.shape}')

## 12. Data Balancing: SMOTE + Undersampling

**Paper approach:**
- SMOTE oversampling of minority class to 10% of majority
- Random undersampling of majority to achieve 1:1 ratio

In [ ]:
# ============================================================
# 12. DATA BALANCING (SMOTE + Undersampling)
# ============================================================
def balance_dataset(X, y):
    """Apply SMOTE oversampling then random undersampling (paper method)."""
    n_majority = (y == 0).sum()
    n_minority = (y == 1).sum()

    print(f'Before balancing: majority={n_majority}, minority={n_minority}')

    if n_minority < 2:
        print('Too few minority samples for SMOTE, using direct oversampling')
        smote_target = max(n_majority // 10, n_minority * 10)
        smote = SMOTE(
            sampling_strategy={1: smote_target},
            k_neighbors=min(n_minority - 1, 5),
            random_state=42
        )
    else:
        smote_target = max(n_majority // 10, n_minority * 2)
        smote = SMOTE(
            sampling_strategy={1: smote_target},
            k_neighbors=min(n_minority - 1, 5),
            random_state=42
        )

    under = RandomUnderSampler(
        sampling_strategy=1.0,
        random_state=42
    )

    pipeline = ImbPipeline([
        ('smote', smote),
        ('undersample', under)
    ])

    X_bal, y_bal = pipeline.fit_resample(X, y)

    print(f'After balancing: class 0={sum(y_bal==0)}, class 1={sum(y_bal==1)}')
    return X_bal, y_bal


# Balance GAE_PDNA embeddings
print('=== Balancing GAE_PDNA embeddings ===')
X_gae_bal, y_gae_bal = balance_dataset(embeddings_gae, labels_np)

print('\n=== Balancing DeepWalk embeddings ===')
X_dw_bal, y_dw_bal = balance_dataset(embeddings_deepwalk, labels_np)

print('\n=== Balancing Node2Vec embeddings ===')
X_n2v_bal, y_n2v_bal = balance_dataset(embeddings_node2vec, labels_np)

## 13. Classification

**Classifiers (from paper Table 8):**
- AdaBoost (base: Decision Tree, 100 estimators)
- Random Forest
- Logistic Regression
- Naive Bayes
- One-Class SVM

In [ ]:
# ============================================================
# 13. CLASSIFICATION ON GAE_PDNA EMBEDDINGS
# ============================================================
def evaluate_classifiers(X, y, embedding_name):
    """Train and evaluate multiple classifiers. Returns results dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    classifiers = {
        'AdaBoost': AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=100, random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced', random_state=42
        ),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, random_state=42
        ),
        'Naive Bayes': GaussianNB(),
    }

    results = {}
    print(f'\n{"="*60}')
    print(f' Classification Results: {embedding_name}')
    print(f'{"="*60}')

    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = (
            clf.predict_proba(X_test)[:, 1]
            if hasattr(clf, 'predict_proba') else y_pred.astype(float)
        )

        prec = precision_score(y_test, y_pred, average='micro')
        rec = recall_score(y_test, y_pred, average='micro')
        f1 = f1_score(y_test, y_pred, average='micro')
        try:
            auc = roc_auc_score(y_test, y_proba)
        except ValueError:
            auc = 0.0

        results[name] = {
            'Precision': prec, 'Recall': rec,
            'F1-Score': f1, 'AUC': auc
        }

        print(f'\n--- {name} ---')
        print(classification_report(y_test, y_pred, digits=2))
        print(f'AUC: {auc:.4f}')

    # One-Class SVM (paper: treats phishing as outliers)
    svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.5)
    svm.fit(X_train[y_train == 0])  # Train on non-phishing only
    y_svm = svm.predict(X_test)
    y_svm_binary = np.where(y_svm == -1, 1, 0)  # outliers = phishing

    prec = precision_score(y_test, y_svm_binary, average='micro')
    rec = recall_score(y_test, y_svm_binary, average='micro')
    f1 = f1_score(y_test, y_svm_binary, average='micro')

    results['One-Class SVM'] = {
        'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'AUC': 0.0
    }

    print(f'\n--- One-Class SVM ---')
    print(classification_report(y_test, y_svm_binary, digits=2))

    return results, X_train, X_test, y_train, y_test


# Evaluate on GAE_PDNA embeddings
results_gae, X_tr_gae, X_te_gae, y_tr_gae, y_te_gae = evaluate_classifiers(
    X_gae_bal, y_gae_bal, 'GAE_PDNA'
)

In [ ]:
# ============================================================
# 14. CLASSIFICATION ON BASELINE EMBEDDINGS
# ============================================================
# DeepWalk
results_dw, _, _, _, _ = evaluate_classifiers(
    X_dw_bal, y_dw_bal, 'DeepWalk'
)

# Node2Vec
results_n2v, _, _, _, _ = evaluate_classifiers(
    X_n2v_bal, y_n2v_bal, 'Node2Vec'
)

## 15. Comparative Analysis

Reproducing Tables 7 and 8 from the paper.

In [ ]:
# ============================================================
# 15. COMPARATIVE ANALYSIS TABLES
# ============================================================

# --- Table 7: Embedding Algorithm Comparison (best classifier each) ---
print('=' * 65)
print(' Table 7: Performance Comparison - Embedding Algorithms')
print(' (Using AdaBoost classifier for each)')
print('=' * 65)

embedding_comparison = {
    'DeepWalk': results_dw.get('AdaBoost', {}),
    'Node2Vec': results_n2v.get('AdaBoost', {}),
    'GAE_PDNA': results_gae.get('AdaBoost', {}),
}

df_embed = pd.DataFrame(embedding_comparison).T
print(df_embed.round(2).to_string())

# --- Table 8: Classifier Comparison (using GAE_PDNA embeddings) ---
print(f'\n{"=" * 65}')
print(' Table 8: Performance Comparison - Classifiers (GAE_PDNA)')
print('=' * 65)

df_clf = pd.DataFrame(results_gae).T
print(df_clf.round(2).to_string())

# --- Table 6: AUC Scores ---
print(f'\n{"=" * 65}')
print(' Table 6: AUC Scores')
print('=' * 65)
if 'AUC' in results_gae.get('AdaBoost', {}):
    print(f'  MagNet:   {magnet_auc * 100:.2f}')
    print(f'  GAE_PDNA: {results_gae["AdaBoost"]["AUC"] * 100:.2f}')

## 16. t-SNE Visualization

2D visualization of GAE_PDNA node embeddings (paper Figure 4).

In [ ]:
# ============================================================
# 16. t-SNE VISUALIZATION OF NODE EMBEDDINGS
# ============================================================
print('Running t-SNE on balanced GAE_PDNA embeddings...')

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_gae_bal)

plt.figure(figsize=(12, 8))

# Non-phishing (blue)
mask_0 = y_gae_bal == 0
plt.scatter(
    X_tsne[mask_0, 0], X_tsne[mask_0, 1],
    c='blue', alpha=0.4, s=10, label='Non-Phishing'
)

# Phishing (red)
mask_1 = y_gae_bal == 1
plt.scatter(
    X_tsne[mask_1, 0], X_tsne[mask_1, 1],
    c='red', alpha=0.6, s=15, label='Phishing'
)

plt.title('t-SNE: GAE_PDNA Node Embeddings (Balanced Dataset)', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('tsne_gae_pdna.png', dpi=300)
plt.show()

print('Saved: tsne_gae_pdna.png')

In [ ]:
# ============================================================
# 17. RESULTS BAR CHART
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Embedding comparison
df_embed_plot = pd.DataFrame(embedding_comparison).T[['Precision', 'Recall', 'F1-Score']]
df_embed_plot.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Embedding Algorithm Comparison (AdaBoost)')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# Chart 2: Classifier comparison on GAE_PDNA
df_clf_plot = pd.DataFrame(results_gae).T[['Precision', 'Recall', 'F1-Score']]
df_clf_plot.plot(kind='bar', ax=axes[1], rot=15)
axes[1].set_title('Classifier Comparison (GAE_PDNA Embeddings)')
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_charts.png', dpi=300)
plt.show()

print('Saved: comparison_charts.png')

In [ ]:
# ============================================================
# 18. SAVE TRAINED MODELS & EMBEDDINGS
# ============================================================
torch.save(model_gae.state_dict(), 'gae_pdna_model.pth')
np.save('embeddings_gae_pdna.npy', embeddings_gae)
np.save('embeddings_deepwalk.npy', embeddings_deepwalk)
np.save('embeddings_node2vec.npy', embeddings_node2vec)

print('Saved:')
print('  gae_pdna_model.pth       - GAE_PDNA model weights')
print('  embeddings_gae_pdna.npy  - GAE_PDNA node embeddings')
print('  embeddings_deepwalk.npy  - DeepWalk node embeddings')
print('  embeddings_node2vec.npy  - Node2Vec node embeddings')

## Summary

### Models Implemented:
| Model | Type | Details |
|-------|------|---------|
| GAE_PDNA | Graph AutoEncoder | 4 PDNConv blocks + decoder, 200 epochs |
| MagNet | Link Prediction | Magnetic Laplacian, 100 epochs |
| DeepWalk | Walk Embedding | Unbiased random walk + Skip-gram |
| Node2Vec | Walk Embedding | Biased random walk + Skip-gram |

### Classifiers:
| Classifier | Notes |
|-----------|-------|
| AdaBoost | Decision tree base, 100 estimators |
| Random Forest | 200 trees, balanced weights |
| Logistic Regression | Standard |
| Naive Bayes | Gaussian |
| One-Class SVM | Outlier detection approach |